In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
import plotly.express as pe
from xgboost import XGBRFClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier,ExtraTreesClassifier,GradientBoostingClassifier,RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report,accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
training_data=pandas.read_csv('../input/mobile-price-classification/train.csv')

In [ ]:
testing_data=pandas.read_csv('../input/mobile-price-classification/train.csv')

In [ ]:
training_data.head()

In [ ]:
testing_data.head()

In [ ]:
training_data.isnull().sum()

In [ ]:
cat_features=list()
num_features=list()
for column_name in training_data.columns:
    unique_values = len(training_data[column_name].unique())
    if unique_values<30:
      cat_features.append(column_name)
    else:
      num_features.append(column_name)

In [ ]:
print("Data with Categorical features =",cat_features)

In [ ]:
print("Data with continuous features =",num_features)

In [ ]:
pe.pie(training_data,names=training_data['blue'].value_counts().index,values=training_data['blue'].value_counts(),
      title='Percentage of mobiles have bluetooth',color=training_data['blue'].value_counts().index)

In [ ]:
pe.pie(training_data,values=training_data.clock_speed.value_counts(),names=training_data.clock_speed.value_counts().index,
      title='speed at which microprocessor executes instructions',hole=0.5)

In [ ]:
pe.pie(training_data,values=training_data.dual_sim.value_counts(),names=training_data.dual_sim.value_counts().index,
      title='Have dual sim')

In [ ]:
pe.pie(training_data,values=training_data.four_g.value_counts(),names=training_data.four_g.value_counts().index,
      title='Has 4G or not',hole=0.5)

In [ ]:
pe.pie(training_data,values=training_data.n_cores.value_counts(),names=training_data.n_cores.value_counts().index,
      title='number of core of processors')

In [ ]:
        pe.pie(training_data,values=training_data.m_dep.value_counts(),names=training_data.m_dep.value_counts().index,
              title='Mobile Depth in cm',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.pc.value_counts(),names=training_data.pc.value_counts().index,
      title='Primary Camera mega pixels',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.sc_h.value_counts(),names=training_data.sc_h.value_counts().index,
      title='Screen Height of mobile in cm',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.sc_w.value_counts(),names=training_data.sc_w.value_counts().index,
      title='Screen Width of mobile in cm',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.talk_time.value_counts(),names=training_data.talk_time.value_counts().index,
      title='Longest time that a single battery charge will last when you are in Hours',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.three_g.value_counts(),names=training_data.three_g.value_counts().index,
      title='has 3G or not',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.touch_screen.value_counts(),names=training_data.touch_screen.value_counts().index,
      title='Has touch screen or not',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.wifi.value_counts(),names=training_data.wifi.value_counts().index,
      title='This is the target variable with value of 0(low cost), 1(medium cost), 2(high cost) and 3(very high cost).',hole=0.6)

In [ ]:
pe.pie(training_data,values=training_data.price_range.value_counts(),names=training_data.price_range.value_counts().index,
      title='This is the target variable with value of 0(low cost), 1(medium cost), 2(high cost) and 3(very high cost).',hole=0.7)

In [ ]:
pe.scatter(training_data,x=training_data.index,y=training_data['battery_power'],color=training_data['battery_power'])

In [ ]:
pe.line(training_data,x=training_data.index,y=training_data['int_memory'],color=training_data['int_memory'])

In [ ]:
pe.histogram(training_data,training_data.index,y=training_data.ram,color=training_data.ram)

In [ ]:
pe.histogram(training_data,training_data.index,y=training_data.mobile_wt,color=training_data.ram)

In [ ]:
pe.area(training_data,x=training_data.index,y=training_data.px_height,color=training_data.px_height)

In [ ]:
pe.scatter(training_data,x=training_data.index, y=training_data.px_width )

In [ ]:
sns.heatmap(training_data)

In [ ]:
sns.displot(training_data.corr())

In [ ]:
pe.scatter_3d(training_data,x=training_data.touch_screen,y=training_data.battery_power,z=training_data.price_range,color=training_data.price_range)

In [ ]:
X=training_data.drop('price_range',axis=1)

In [ ]:
y=training_data['price_range']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
xc=XGBRFClassifier(base_score=0.5, booster='gbtree', callbacks=None,
                colsample_bylevel=1, colsample_bytree=1,
                early_stopping_rounds=None, enable_categorical=False,
                eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
                importance_type=None, interaction_constraints='', max_bin=256,
                max_cat_to_onehot=4, max_delta_step=0, max_depth=6,
                max_leaves=0, min_child_weight=1,
                monotone_constraints='()', n_estimators=100, n_jobs=0,
                num_parallel_tree=100, objective='binary:logistic',
                predictor='auto', random_state=0, reg_alpha=0,
                sampling_method='uniform', scale_pos_weight=1)

In [ ]:
xc.fit(X_train,y_train)

In [ ]:
importances = pandas.DataFrame(data={
    'Attribute': X_train.columns,
    'Importance': xc.feature_importances_
})
importances = importances.sort_values(by='Importance', ascending=False)

In [ ]:
plt.bar(x=importances['Attribute'], height=importances['Importance'], color='#087E8B')
plt.title('Feature importances obtained from coefficients', size=20)
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
classifier=[
    
    LogisticRegression(),
    DecisionTreeClassifier(),
    SVC(),
    AdaBoostClassifier(),
    ExtraTreesClassifier(),
    GradientBoostingClassifier(),
    RandomForestClassifier(),
    GaussianNB(),
    KNeighborsClassifier(),
    XGBRFClassifier()
    
]

In [ ]:
testing_data=testing_data.drop('price_range',axis=1)

In [ ]:
testing_data=testing_data.head(660)

In [ ]:
name=[],
score=[],
models=[],
i=0
for m in classifier:
    m.fit(X_train, y_train)
    y_pred = m.predict(testing_data)

    print(f'model: {str(m)}')
    print(classification_report(y_test,y_pred, zero_division=1))
    print('-'*30, '\n')

In [ ]:
name=[],
score=[],
models=[],
i=0
for m in classifier:
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)

    print(f'model: {str(m)}')
    print(accuracy_score(y_test,y_pred
))
    print('-'*30, '\n')

Support Vector Classifier performed best on this dataset